In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from app.data_providers import clean_source_dataframe, get_shots_dataframe
from app.config import CLEAN_SOURCE_COLUMNS, PLAYER_CHOICE

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.options.display.max_columns = None

In [2]:
df = get_shots_dataframe()

Repo card metadata block was not found. Setting CardData to empty.


# Nans for Target variable

In [3]:
target_colums = ['SHOT_MADE_FLAG', 'shotResult', 'actionType']

In [4]:
for target in target_colums:
    print(f'{target} Nan values: ', df[target].isna().sum())

SHOT_MADE_FLAG Nan values:  724
shotResult Nan values:  359710
actionType Nan values:  359710


In [5]:
# test if other columns are not none for these
df.loc[df['SHOT_MADE_FLAG'].isna(), target_colums].isna().sum()

SHOT_MADE_FLAG    724
shotResult          0
actionType          0
dtype: int64

We can use the other target columns to correct the nan values

In [6]:
mask = df['SHOT_MADE_FLAG'].isna()
df.loc[mask, 'SHOT_MADE_FLAG'] = df.loc[mask, 'shotResult'].map({'Made': 1, 'Missed': 0})

df['SHOT_MADE_FLAG'].isna().sum()

np.int64(0)

In [8]:
# Comparison of the target columns, to make sure they are consistent
df['shotResult'] = df['shotResult'].replace({'Made': 1, 'Missed': 0})
df['actionType'] = df['actionType'].replace({'Made Shot': 1, 'Made Shot                               ': 1, 'Missed Shot': 0, 'Missed Shot                             ': 0})

In [9]:
mask = ~(df['shotResult'].isna())
sum(df.loc[mask, 'SHOT_MADE_FLAG'] != df.loc[mask, 'shotResult'])

16

In [10]:
mask = ~(df['shotResult'].isna())
sum((df.loc[mask, 'SHOT_MADE_FLAG'] != df.loc[mask, 'actionType']) & (df.loc[mask, 'actionType'] != 'Free Throw'))

16

In [11]:
df.loc[(df['SHOT_MADE_FLAG'] != df['shotResult']) & ~(df['shotResult'].isna())].head()

,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,actionId,gameId,GRID_TYPE,GAME_ID_x,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD_x,MINUTES_REMAINING,SECONDS_REMAINING,EVENT_TYPE,ACTION_TYPE,SHOT_TYPE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM,GAME_ID_y,EVENTNUM,EVENTMSGTYPE,EVENTMSGACTIONTYPE,PERIOD_y,WCTIMESTRING,PCTIMESTRING,HOMEDESCRIPTION,NEUTRALDESCRIPTION,VISITORDESCRIPTION,SCORE,SCOREMARGIN,PERSON1TYPE,PLAYER1_ID,PLAYER1_NAME,PLAYER1_TEAM_ID,PLAYER1_TEAM_CITY,PLAYER1_TEAM_NICKNAME,PLAYER1_TEAM_ABBREVIATION,PERSON2TYPE,PLAYER2_ID,PLAYER2_NAME,PLAYER2_TEAM_ID,PLAYER2_TEAM_CITY,PLAYER2_TEAM_NICKNAME,PLAYER2_TEAM_ABBREVIATION,PERSON3TYPE,PLAYER3_ID,PLAYER3_NAME,PLAYER3_TEAM_ID,PLAYER3_TEAM_CITY,PLAYER3_TEAM_NICKNAME,PLAYER3_TEAM_ABBREVIATION,VIDEO_AVAILABLE_FLAG,year,is_playoffs,shotValue
5905021,43,PT08M47.00S,1,1610612761,TOR,200768,Lowry,K. Lowry,63,253,26,0,1,NaN,NaN,0,v,MISS Lowry 26' 3PT Jump Shot,0,Jump Shot,1,34,21700211,Shot Chart Detail,21700211.0,43.0,203076.0,Anthony Davis,1.610613e+09,New Orleans Pelicans,1.0,8.0,53.0,Made Shot,Tip Dunk Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,0.0,0.0,-6.0,1.0,1.0,20171115.0,NOP,TOR,21700211.0,43.0,2.0,1.0,1.0,8:16 PM,8:47,NaN,NaN,MISS Lowry 26' 3PT Jump Shot,NaN,NaN,5.0,200768.0,Kyle Lowry,1.610613e+09,Toronto,Raptors,TOR,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1.0,2017,False,NaN
5905031,70,PT06M52.00S,1,1610612761,TOR,201942,DeRozan,D. DeRozan,230,73,24,1,1,16.0,9.0,25,v,DeRozan 24' 3PT Jump Shot (3 PTS) (Lowry 1 AST),1,Jump Shot,1,57,21700211,Shot Chart Detail,21700211.0,70.0,200765.0,Rajon Rondo,1.610613e+09,New Orleans Pelicans,1.0,7.0,14.0,Missed Shot,Layup Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,1.0,16.0,0.0,1.0,0.0,20171115.0,NOP,TOR,21700211.0,70.0,1.0,1.0,1.0,8:22 PM,6:52,NaN,NaN,DeRozan 24' 3PT Jump Shot (3 PTS) (Lowry 1 AST),9 - 16,7,5.0,201942.0,DeMar DeRozan,1.610613e+09,Toronto,Raptors,TOR,5.0,200768.0,Kyle Lowry,1.610613e+09,Toronto,Raptors,TOR,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1.0,2017,False,NaN
5905044,109,PT04M03.00S,1,1610612740,NOP,201950,Holiday,J. Holiday,2,73,7,1,1,27.0,16.0,43,h,Holiday 7' Floating Jump Shot (4 PTS),1,Floating Jump shot,1,83,21700211,Shot Chart Detail,21700211.0,109.0,203121.0,Darius Miller,1.610613e+09,New Orleans Pelicans,1.0,4.0,28.0,Missed Shot,Jump Shot,2PT Field Goal,Mid-Range,Right Side(R),16-24 ft.,16.0,167.0,21.0,1.0,0.0,20171115.0,NOP,TOR,21700211.0,109.0,1.0,78.0,1.0,8:26 PM,4:03,Holiday 7' Floating Jump Shot (4 PTS),NaN,NaN,16 - 27,11,4.0,201950.0,Jrue Holiday,1.610613e+09,New Orleans,Pelicans,NOP,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1.0,2017,False,NaN
5905049,128,PT02M36.00S,1,1610612740,NOP,203076,Davis,A. Davis,-37,232,23,0,1,NaN,NaN,0,h,MISS Davis 23' Jump Shot,0,Jump Shot,1,96,21700211,Shot Chart Detail,21700211.0,128.0,203076.0,Anthony Davis,1.610613e+09,New Orleans Pelicans,1.0,3.0,10.0,Made Shot,Jump Shot,3PT Field Goal,Above the Break 3,Center(C),24+ ft.,25.0,8.0,258.0,1.0,1.0,20171115.0,NOP,TOR,21700211.0,128.0,2.0,1.0,1.0,10:18 PM,2:36,MISS Davis 23' Jump Shot,NaN,NaN,NaN,NaN,4.0,203076.0,Anthony Davis,1.610613e+09,New Orleans,Pelicans,NOP,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1.0,2017,False,NaN
5905060,158,PT00M00.20S,1,1610612761,TOR,101139,Miles,C. Miles,-148,203,25,0,1,NaN,NaN,0,v,MISS Miles 25' 3PT Jump Shot,0,Jump Shot,1,119,21700211,Shot Chart Detail,21700211.0,158.0,101139.0,CJ Miles,1.610613e+09,Toronto Raptors,1.0,0.0,27.0,Made Shot,Dunk Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,0.0,5.0,3.0,1.0,1.0,20171115.0,NOP,TOR,21700211.0,158.0,2.0,1.0,1.0,10:18 PM,0:00,NaN,NaN,MISS Miles 25' 3PT Jump Shot,NaN,NaN,5.0,101139.0,CJ Miles,1.610613

16 rows are inconsistent between the different target variables, we drop these

In [12]:
df = df[~(df['shotResult'].notna() & (df['SHOT_MADE_FLAG'] != df['shotResult']))]

mask = df['shotResult'].notna()
sum(df.loc[mask, 'SHOT_MADE_FLAG'] != df.loc[mask, 'shotResult'])

0

# Check other relevant columns

In [13]:
df[CLEAN_SOURCE_COLUMNS].isna().sum()

SHOT_DISTANCE                    724
SHOT_ZONE_RANGE                  724
SHOT_ZONE_BASIC                  724
SHOT_ZONE_AREA                   724
LOC_X                            724
LOC_Y                            724
ACTION_TYPE                      724
HTM                          1756728
PLAYER1_TEAM_ABBREVIATION        719
PLAYER_ID                        724
scoreHome                    1792919
scoreAway                    1792919
SHOT_MADE_FLAG                     0
dtype: int64

In [14]:
df.loc[df['PLAYER_NAME'].isin(PLAYER_CHOICE), CLEAN_SOURCE_COLUMNS].isna().sum()

SHOT_DISTANCE                     0
SHOT_ZONE_RANGE                   0
SHOT_ZONE_BASIC                   0
SHOT_ZONE_AREA                    0
LOC_X                             0
LOC_Y                             0
ACTION_TYPE                       0
HTM                          146099
PLAYER1_TEAM_ABBREVIATION         1
PLAYER_ID                         0
scoreHome                    103033
scoreAway                    103033
SHOT_MADE_FLAG                    0
dtype: int64

None of the 724 nans are in the player choice, we are investigating. We'll just drop these rows

In [15]:
df = df.dropna(subset=['SHOT_DISTANCE', 'SHOT_ZONE_RANGE', 'SHOT_ZONE_BASIC', 'LOC_X', 'LOC_Y'])
df[CLEAN_SOURCE_COLUMNS].isna().sum()

SHOT_DISTANCE                      0
SHOT_ZONE_RANGE                    0
SHOT_ZONE_BASIC                    0
SHOT_ZONE_AREA                     0
LOC_X                              0
LOC_Y                              0
ACTION_TYPE                        0
HTM                          1756004
PLAYER1_TEAM_ABBREVIATION        716
PLAYER_ID                          0
scoreHome                    1792782
scoreAway                    1792782
SHOT_MADE_FLAG                     0
dtype: int64

In [16]:
# Check nans introduced by free throws
df.loc[df['SHOT_TYPE'] == '1PT Free Throw', CLEAN_SOURCE_COLUMNS].isna().sum()

SHOT_DISTANCE                      0
SHOT_ZONE_RANGE                    0
SHOT_ZONE_BASIC                    0
SHOT_ZONE_AREA                     0
LOC_X                              0
LOC_Y                              0
ACTION_TYPE                        0
HTM                          1756004
PLAYER1_TEAM_ABBREVIATION        137
PLAYER_ID                          0
scoreHome                     171395
scoreAway                     171395
SHOT_MADE_FLAG                     0
dtype: int64

Most of the Nans are present in rows with a free throw. This includes HTM, PLAYER1_TEAM_ABBREVIATION, scoreHome, scoreAway

In [17]:
# HTM and VTM can be filled by checking other rows, with the same gameid, that are not nan
df['HTM'] = df.groupby('GAME_ID_x')['HTM'].transform('first')
df['VTM'] = df.groupby('GAME_ID_x')['VTM'].transform('first')
df.loc[df['SHOT_TYPE'] == '1PT Free Throw', CLEAN_SOURCE_COLUMNS].isna().sum()

SHOT_DISTANCE                     0
SHOT_ZONE_RANGE                   0
SHOT_ZONE_BASIC                   0
SHOT_ZONE_AREA                    0
LOC_X                             0
LOC_Y                             0
ACTION_TYPE                       0
HTM                               0
PLAYER1_TEAM_ABBREVIATION       137
PLAYER_ID                         0
scoreHome                    171395
scoreAway                    171395
SHOT_MADE_FLAG                    0
dtype: int64

In [18]:
# We can fill PLAYER1_TEAM_ABBREVIATION, by checking other rows with same player and team id
df['PLAYER1_TEAM_ABBREVIATION'] = df.groupby(['PLAYER_ID', 'TEAM_ID'])['PLAYER1_TEAM_ABBREVIATION'].transform('first')

In [22]:
# Handle nans in scores (Mykolav is handling it, comment out for now)
#
#def update_scores(group):
#    # start from previous non-zero scores
#    group['scoreHome'] = group['scoreHome'].replace(0, np.nan).ffill().fillna(0)
#    group['scoreAway'] = group['scoreAway'].replace(0, np.nan).ffill().fillna(0)
#
#    for i in range(1, len(group)):
#        prev_home = group.iloc[i-1]['scoreHome']
#        prev_away = group.iloc[i-1]['scoreAway']
#
#        if group.iloc[i]['SHOT_MADE_FLAG'] == 1:
#            if group.iloc[i]['HTM'] == group.iloc[i]['PLAYER1_TEAM_ABBREVIATION']:
#                group.iloc[i, group.columns.get_loc('scoreHome')] = prev_home + 1
#                group.iloc[i, group.columns.get_loc('scoreAway')] = prev_away
#            else:
#                group.iloc[i, group.columns.get_loc('scoreHome')] = prev_home
#                group.iloc[i, group.columns.get_loc('scoreAway')] = prev_away + 1
#        else:
#            group.iloc[i, group.columns.get_loc('scoreHome')] = prev_home
#            group.iloc[i, group.columns.get_loc('scoreAway')] = prev_away
#
#    return group

# Check nans for additional columns

In [19]:
additional_clean_cols = [
    'PLAYER_NAME',
    'TEAM_ID',
    'PERIOD_x',
    'MINUTES_REMAINING',
    'SECONDS_REMAINING',
    'GAME_DATE',
    'is_playoffs'
]

CLEAN_SOURCE_COLUMNS_extended = CLEAN_SOURCE_COLUMNS.copy()
CLEAN_SOURCE_COLUMNS_extended.extend(additional_clean_cols)

In [20]:
df[CLEAN_SOURCE_COLUMNS_extended].isna().sum()

SHOT_DISTANCE                      0
SHOT_ZONE_RANGE                    0
SHOT_ZONE_BASIC                    0
SHOT_ZONE_AREA                     0
LOC_X                              0
LOC_Y                              0
ACTION_TYPE                        0
HTM                                0
PLAYER1_TEAM_ABBREVIATION          5
PLAYER_ID                          0
scoreHome                    1792782
scoreAway                    1792782
SHOT_MADE_FLAG                     0
PLAYER_NAME                      161
TEAM_ID                            0
PERIOD_x                           0
MINUTES_REMAINING            1756009
SECONDS_REMAINING            1756009
GAME_DATE                    1756004
is_playoffs                        0
dtype: int64

In [24]:
# extract minutes and seconds remaining from clock column and fill the nans
extracted = df['clock'].str.extract(r'PT(\d+)M(\d+)\.')

# fill only NaNs
df['MINUTES_REMAINING'] = df['MINUTES_REMAINING'].fillna(extracted[0].astype(float))
df['SECONDS_REMAINING'] = df['SECONDS_REMAINING'].fillna(extracted[1].astype(float))
df[CLEAN_SOURCE_COLUMNS_extended].isna().sum()

SHOT_DISTANCE                      0
SHOT_ZONE_RANGE                    0
SHOT_ZONE_BASIC                    0
SHOT_ZONE_AREA                     0
LOC_X                              0
LOC_Y                              0
ACTION_TYPE                        0
HTM                                0
PLAYER1_TEAM_ABBREVIATION          5
PLAYER_ID                          0
scoreHome                    1792782
scoreAway                    1792782
SHOT_MADE_FLAG                     0
PLAYER_NAME                        0
TEAM_ID                            0
PERIOD_x                           0
MINUTES_REMAINING                  0
SECONDS_REMAINING                  0
GAME_DATE                          0
is_playoffs                        0
dtype: int64

In [25]:
# Fill the game date nans via other rows with the same game id
df['GAME_DATE'] = df.groupby('GAME_ID_x')['GAME_DATE'].transform('first')
df[CLEAN_SOURCE_COLUMNS_extended].isna().sum()

SHOT_DISTANCE                      0
SHOT_ZONE_RANGE                    0
SHOT_ZONE_BASIC                    0
SHOT_ZONE_AREA                     0
LOC_X                              0
LOC_Y                              0
ACTION_TYPE                        0
HTM                                0
PLAYER1_TEAM_ABBREVIATION          5
PLAYER_ID                          0
scoreHome                    1792782
scoreAway                    1792782
SHOT_MADE_FLAG                     0
PLAYER_NAME                        0
TEAM_ID                            0
PERIOD_x                           0
MINUTES_REMAINING                  0
SECONDS_REMAINING                  0
GAME_DATE                          0
is_playoffs                        0
dtype: int64

In [26]:
display(df.loc[df['PLAYER1_TEAM_ABBREVIATION'].isna()])

,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,actionId,gameId,GRID_TYPE,GAME_ID_x,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD_x,MINUTES_REMAINING,SECONDS_REMAINING,EVENT_TYPE,ACTION_TYPE,SHOT_TYPE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM,GAME_ID_y,EVENTNUM,EVENTMSGTYPE,EVENTMSGACTIONTYPE,PERIOD_y,WCTIMESTRING,PCTIMESTRING,HOMEDESCRIPTION,NEUTRALDESCRIPTION,VISITORDESCRIPTION,SCORE,SCOREMARGIN,PERSON1TYPE,PLAYER1_ID,PLAYER1_NAME,PLAYER1_TEAM_ID,PLAYER1_TEAM_CITY,PLAYER1_TEAM_NICKNAME,PLAYER1_TEAM_ABBREVIATION,PERSON2TYPE,PLAYER2_ID,PLAYER2_NAME,PLAYER2_TEAM_ID,PLAYER2_TEAM_CITY,PLAYER2_TEAM_NICKNAME,PLAYER2_TEAM_ABBREVIATION,PERSON3TYPE,PLAYER3_ID,PLAYER3_NAME,PLAYER3_TEAM_ID,PLAYER3_TEAM_CITY,PLAYER3_TEAM_NICKNAME,PLAYER3_TEAM_ABBREVIATION,VIDEO_AVAILABLE_FLAG,year,is_playoffs,shotValue
2446365,541,PT00M00.00S,4,0,NaN,1610612754,NaN,NaN,0,15,15,1,0,0.0,0.0,0,h,PACERS Free Throw,Free Throw,Free Throw 1 of 2,0,535,20500121,NaN,20500121.0,541.0,1.610613e+09,Brevin Knight,0.000000e+00,NaN,4.0,0.0,0.0,Made Shot,Free Throw,1PT Free Throw,Mid-Range,Center(C),8-16 ft.,15.0,0.0,15.0,1.0,1.0,20051118.0,IND,CHA,20500121.0,541.0,3.0,11.0,4.0,9:52 PM,0:00,PACERS Free Throw,NaN,NaN,NaN,NaN,2.0,1.610613e+09,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,2005,False,NaN
6707955,585,PT07M12.00S,4,1610612743,DEN,1629076,Cook,T. Cook,-20,18,3,0,1,NaN,NaN,0,v,MISS Cook 3' Layup,0,Layup Shot,1,413,21901318,Shot Chart Detail,21901318.0,585.0,1.629076e+06,Nikola Jokic,1.610613e+09,Denver Nuggets,4.0,7.0,12.0,Missed Shot,Layup Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,2.0,-20.0,18.0,1.0,0.0,20200814.0,TOR,DEN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,False,NaN
6707962,572,PT05M30.00S,4,1610612743,DEN,1629076,Cook,T. Cook,0,15,15,1,0,103.0,93.0,196,v,Cook Free Throw 1 of 2 (1 PTS),Free Throw,Free Throw 1 of 2,1,427,21901318,NaN,21901318.0,572.0,1.629076e+06,Nikola Jokic,1.610613e+09,NaN,4.0,5.0,30.0,Made Shot,Free Throw,1PT Free Throw,Mid-Range,Center(C),8-16 ft.,15.0,0.0,15.0,1.0,1.0,20200814.0,TOR,DEN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,False,NaN
6707963,577,PT05M30.00S,4,1610612743,DEN,1629076,Cook,T. Cook,0,15,15,1,0,103.0,94.0,197,v,Cook Free Throw 2 of 2 (2 PTS),Free Throw,Free Throw 2 of 2,1,430,21901318,NaN,21901318.0,577.0,1.629076e+06,Nikola Jokic,1.610613e+09,NaN,4.0,5.0,30.0,Made Shot,Free Throw,1PT Free Throw,Mid-Range,Center(C),8-16 ft.,15.0,0.0,15.0,1.0,1.0,20200814.0,TOR,DEN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,False,NaN
6707982,621,PT01M42.00S,4,1610612743,DEN,1629076,Cook,T. Cook,0,-6,1,1,1,112.0,104.0,216,v,Cook 1' Tip Dunk Shot (4 PTS),1,Tip Dunk Shot,1,462,21901318,Shot Chart Detail,21901318.0,621.0,1.629076e+06,Nikola Jokic,1.610613e+09,Denver Nuggets,4.0,1.0,42.0,Made Shot,Tip Dunk Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,0.0,0.0,-6.0,1.0,1.0,20200814.0,TOR,DEN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,False,NaN


In [27]:
# We just drop the rows, since there are more nans and we're not intereseted in Brevin Knight and T.Cook
df = df.dropna(subset=['PLAYER1_TEAM_ABBREVIATION'])

In [28]:
df[CLEAN_SOURCE_COLUMNS_extended].isna().sum()

SHOT_DISTANCE                      0
SHOT_ZONE_RANGE                    0
SHOT_ZONE_BASIC                    0
SHOT_ZONE_AREA                     0
LOC_X                              0
LOC_Y                              0
ACTION_TYPE                        0
HTM                                0
PLAYER1_TEAM_ABBREVIATION          0
PLAYER_ID                          0
scoreHome                    1792781
scoreAway                    1792781
SHOT_MADE_FLAG                     0
PLAYER_NAME                        0
TEAM_ID                            0
PERIOD_x                           0
MINUTES_REMAINING                  0
SECONDS_REMAINING                  0
GAME_DATE                          0
is_playoffs                        0
dtype: int64

# Check duplicated game events

In [29]:
df.loc[df.duplicated(subset=['GAME_ID_x', 'GAME_EVENT_ID'], keep=False)].head(10)

,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,actionId,gameId,GRID_TYPE,GAME_ID_x,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD_x,MINUTES_REMAINING,SECONDS_REMAINING,EVENT_TYPE,ACTION_TYPE,SHOT_TYPE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM,GAME_ID_y,EVENTNUM,EVENTMSGTYPE,EVENTMSGACTIONTYPE,PERIOD_y,WCTIMESTRING,PCTIMESTRING,HOMEDESCRIPTION,NEUTRALDESCRIPTION,VISITORDESCRIPTION,SCORE,SCOREMARGIN,PERSON1TYPE,PLAYER1_ID,PLAYER1_NAME,PLAYER1_TEAM_ID,PLAYER1_TEAM_CITY,PLAYER1_TEAM_NICKNAME,PLAYER1_TEAM_ABBREVIATION,PERSON2TYPE,PLAYER2_ID,PLAYER2_NAME,PLAYER2_TEAM_ID,PLAYER2_TEAM_CITY,PLAYER2_TEAM_NICKNAME,PLAYER2_TEAM_ABBREVIATION,PERSON3TYPE,PLAYER3_ID,PLAYER3_NAME,PLAYER3_TEAM_ID,PLAYER3_TEAM_CITY,PLAYER3_TEAM_NICKNAME,PLAYER3_TEAM_ABBREVIATION,VIDEO_AVAILABLE_FLAG,year,is_playoffs,shotValue
21,47,PT06M14.00S,1,1610612741,CHI,26,Longley,L. Longley,0,0,0,0,1,0.0,0.0,0,v,MISS Longley Layup,0,Layup Shot,0,49,29600001,Shot Chart Detail,29600001.0,47.0,26.0,Dennis Rodman,1.610613e+09,Chicago Bulls,1.0,6.0,14.0,Missed Shot,Layup Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,0.0,0.0,0.0,1.0,0.0,19961101.0,BOS,CHI,29600001.0,47.0,2.0,5.0,1.0,11:27 PM,6:14,Ellison BLOCK (1 BLK),NaN,MISS Longley Layup,NaN,NaN,5.0,26.0,Luc Longley,1.610613e+09,Chicago,Bulls,CHI,0.0,0.0,NaN,NaN,NaN,NaN,NaN,4.0,442.0,Pervis Ellison,1.610613e+09,Boston,Celtics,BOS,0.0,1996,False,NaN
22,47,PT06M14.00S,1,1610612738,BOS,442,Ellison,P. Ellison,0,0,0,NaN,0,0.0,0.0,0,h,Ellison BLOCK (1 BLK),NaN,NaN,0,50,29600001,Shot Chart Detail,29600001.0,47.0,26.0,Dennis Rodman,1.610613e+09,Chicago Bulls,1.0,6.0,14.0,Missed Shot,Layup Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,0.0,0.0,0.0,1.0,0.0,19961101.0,BOS,CHI,29600001.0,47.0,2.0,5.0,1.0,11:27 PM,6:14,Ellison BLOCK (1 BLK),NaN,MISS Longley Layup,NaN,NaN,5.0,26.0,Luc Longley,1.610613e+09,Chicago,Bulls,CHI,0.0,0.0,NaN,NaN,NaN,NaN,NaN,4.0,442.0,Pervis Ellison,1.610613e+09,Boston,Celtics,BOS,0.0,1996,False,NaN
33,76,PT02M39.00S,1,1610612738,BOS,103,Day,T. Day,0,0,0,0,1,0.0,0.0,0,h,MISS Day Layup,0,Layup Shot,0,80,29600001,Shot Chart Detail,29600001.0,76.0,103.0,Dennis Rodman,1.610613e+09,Boston Celtics,1.0,2.0,39.0,Missed Shot,Layup Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,0.0,0.0,0.0,1.0,0.0,19961101.0,BOS,CHI,29600001.0,76.0,2.0,5.0,1.0,11:36 PM,2:39,MISS Day Layup,NaN,Wennington BLOCK (1 BLK),NaN,NaN,4.0,103.0,Todd Day,1.610613e+09,Boston,Celtics,BOS,0.0,0.0,NaN,NaN,NaN,NaN,NaN,5.0,82.0,Bill Wennington,1.610613e+09,Chicago,Bulls,CHI,0.0,1996,False,NaN
34,76,PT02M39.00S,1,1610612741,CHI,82,Wennington,B. Wennington,0,0,0,NaN,0,0.0,0.0,0,v,Wennington BLOCK (1 BLK),NaN,NaN,0,81,29600001,Shot Chart Detail,29600001.0,76.0,103.0,Dennis Rodman,1.610613e+09,Boston Celtics,1.0,2.0,39.0,Missed Shot,Layup Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,0.0,0.0,0.0,1.0,0.0,19961101.0,BOS,CHI,29600001.0,76.0,2.0,5.0,1.0,11:36 PM,2:39,MISS Day Layup,NaN,Wennington BLOCK (1 BLK),NaN,NaN,4.0,103.0,Todd Day,1.610613e+09,Boston,Celtics,BOS,0.0,0.0,NaN,NaN,NaN,NaN,NaN,5.0,82.0,Bill Wennington,1.610613e+09,Chicago,Bulls,CHI,0.0,1996,False,NaN
36,81,PT02M08.00S,1,1610612738,BOS,296,Fox,R. Fox,0,0,0,0,1,0.0,0.0,0,h,MISS Fox Layup,0,Layup Shot,0,85,29600001,Shot Chart Detail,29600001.0,81.0,296.0,Dennis Rodman,1.610613e+09,Boston Celtics,1.0,2.0,8.0,Missed Shot,Layup Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,0.0,0.0,0.0,1.0,0.0,19961101.0,BOS,CHI,29600001.0,81.0,2.0,5.0,1.0,11:36 PM,2:08,MISS Fox Layup,NaN,Kukoc BLOCK (1 BLK),NaN,NaN,4.0,296.0,Rick Fox,1.610613e+09,Boston,Celtics,BOS,0.0,0.0,NaN,NaN,NaN,NaN,NaN,5.0,389.0,Toni Kukoc,1.610613e+09,Chicago,Bulls,CHI,0.0,1996,False,NaN
37,81,PT02M08.

We can see that for the duplicated events there is always the shot attempt and in a second row a block. We do not need the second row, since it does not add
additional information. We already have information about the block int the description of the first row. Therefore we can drop the duplicated rows with the block.
We can do this by filtering via the NaN values in shotResult 

In [30]:
df = df[~((df.duplicated(subset=['GAME_ID_x', 'GAME_EVENT_ID'], keep=False)) & df['shotResult'].isna())]
df.duplicated(subset=['GAME_ID_x', 'GAME_EVENT_ID']).sum()

np.int64(556)

In [31]:
df.loc[df.duplicated(subset=['GAME_ID_x', 'GAME_EVENT_ID'], keep=False)].head(10)

,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,actionId,gameId,GRID_TYPE,GAME_ID_x,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD_x,MINUTES_REMAINING,SECONDS_REMAINING,EVENT_TYPE,ACTION_TYPE,SHOT_TYPE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM,GAME_ID_y,EVENTNUM,EVENTMSGTYPE,EVENTMSGACTIONTYPE,PERIOD_y,WCTIMESTRING,PCTIMESTRING,HOMEDESCRIPTION,NEUTRALDESCRIPTION,VISITORDESCRIPTION,SCORE,SCOREMARGIN,PERSON1TYPE,PLAYER1_ID,PLAYER1_NAME,PLAYER1_TEAM_ID,PLAYER1_TEAM_CITY,PLAYER1_TEAM_NICKNAME,PLAYER1_TEAM_ABBREVIATION,PERSON2TYPE,PLAYER2_ID,PLAYER2_NAME,PLAYER2_TEAM_ID,PLAYER2_TEAM_CITY,PLAYER2_TEAM_NICKNAME,PLAYER2_TEAM_ABBREVIATION,PERSON3TYPE,PLAYER3_ID,PLAYER3_NAME,PLAYER3_TEAM_ID,PLAYER3_TEAM_CITY,PLAYER3_TEAM_NICKNAME,PLAYER3_TEAM_ABBREVIATION,VIDEO_AVAILABLE_FLAG,year,is_playoffs,shotValue
4973322,556,PT00M40.30S,4,1610612751,BKN,203928,Jefferson,C. Jefferson,0,1,0,1,1,121.0,103.0,224,v,Jefferson Dunk (8 PTS) (Jack 2 AST),1,Dunk Shot,1,488,21400006,Shot Chart Detail,21400006.0,556.0,203928.0,Kevin Garnett,1.610613e+09,Brooklyn Nets,4.0,0.0,40.0,Made Shot,Dunk Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,0.0,0.0,1.0,1.0,1.0,20141029.0,BOS,BKN,21400006.0,556.0,1.0,7.0,4.0,9:54 PM,0:40,NaN,NaN,Jefferson Dunk (8 PTS) (Jack 2 AST),103 - 121,18,5.0,203928.0,Cory Jefferson,1.610613e+09,Brooklyn,Nets,BKN,5.0,101127.0,Jarrett Jack,1.610613e+09,Brooklyn,Nets,BKN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1.0,2014,False,NaN
4973323,556,PT00M40.30S,4,1610612751,BKN,203928,Jefferson,C. Jefferson,0,1,0,1,1,121.0,103.0,224,v,Jefferson Dunk (8 PTS) (Jack 2 AST),1,Dunk Shot,1,488,21400006,Shot Chart Detail,21400006.0,556.0,203928.0,Kevin Garnett,1.610613e+09,Brooklyn Nets,4.0,0.0,40.0,Made Shot,Dunk Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,0.0,0.0,1.0,1.0,1.0,20141029.0,BOS,BKN,21400006.0,556.0,1.0,7.0,4.0,9:54 PM,0:40,NaN,NaN,Jefferson Dunk (8 PTS) (Jack 2 AST),103 - 121,18,5.0,203928.0,Cory Jefferson,1.610613e+09,Brooklyn,Nets,BKN,5.0,101127.0,Jarrett Jack,1.610613e+09,Brooklyn,Nets,BKN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1.0,2014,False,NaN
4982706,61,PT05M07.00S,1,1610612754,IND,203524,Hill,S. Hill,-155,-10,16,1,1,10.0,10.0,20,h,S. Hill 16' Jump Shot (5 PTS) (Scola 1 AST),1,Jump Shot,1,65,21400049,Shot Chart Detail,21400049.0,61.0,203524.0,Solomon Hill,1.610613e+09,Indiana Pacers,1.0,5.0,7.0,Made Shot,Jump Shot,2PT Field Goal,Mid-Range,Left Side(L),8-16 ft.,15.0,-155.0,-10.0,1.0,1.0,20141104.0,IND,MIL,21400049.0,61.0,1.0,1.0,1.0,7:23 PM,5:07,S. Hill 16' Jump Shot (5 PTS) (Scola 1 AST),NaN,NaN,10 - 10,TIE,4.0,203524.0,Solomon Hill,1.610613e+09,Indiana,Pacers,IND,4.0,2449.0,Luis Scola,1.610613e+09,Indiana,Pacers,IND,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1.0,2014,False,NaN
4982707,61,PT05M07.00S,1,1610612754,IND,203524,Hill,S. Hill,-155,-10,16,1,1,10.0,10.0,20,h,S. Hill 16' Jump Shot (5 PTS) (Scola 1 AST),1,Jump Shot,1,65,21400049,Shot Chart Detail,21400049.0,61.0,203524.0,Solomon Hill,1.610613e+09,Indiana Pacers,1.0,5.0,7.0,Made Shot,Jump Shot,2PT Field Goal,Mid-Range,Left Side(L),8-16 ft.,15.0,-155.0,-10.0,1.0,1.0,20141104.0,IND,MIL,21400049.0,61.0,1.0,1.0,1.0,7:23 PM,5:07,S. Hill 16' Jump Shot (5 PTS) (Scola 1 AST),NaN,NaN,10 - 10,TIE,4.0,203524.0,Solomon Hill,1.610613e+09,Indiana,Pacers,IND,4.0,2449.0,Luis Scola,1.610613e+09,Indiana,Pacers,IND,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1.0,2014,False,NaN
4985573,155,PT08M33.00S,2,1610612741,CHI,202703,Mirotic,N. Mirotic,-8,9,1,1,1,29.0,30.0,59,v,Mirotic 1' Layup (2 PTS) (Gibson 1 AST),1,Layup Shot,1,146,21400062,Shot Chart Detail,21400062.0,155.0,202703.0,Pau Gasol,1.610613e+09,Chicago Bulls,2.0,8.0,33.0,Made Shot,Layup Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,1.0,-8.0,9.0,1.0,1.0,20141105.0,MIL,CHI,21400062.0,155

In [32]:
df.duplicated().sum()

np.int64(556)

The remaining duplicates have the same value in all of the rows, we can just drop one

In [33]:
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
df.drop(['shotResult', 'actionType'], axis=1).to_parquet('../data/clean_firstVersion.parquet')